# Qwen2.5-1.5B + SFT adapter, 30-question hallucination check

Same 30 questions as a subset of `results/rag_e2e/norag.jsonl` (TinyLlama SFT, already judged), same prompt as the
agent's norag path (system prompt + question) and the same sampling (temp 0.8, top_p 0.9, top_k 30, rep 1.05, 160 tokens).
Upload `results/qwen_norag_questions.jsonl` to `DRIVE_DIR`; the answers are written next to it.

Set IN_FILE / OUT_FILE / COND in the first code cell (norag or rag).


In [ ]:
!pip install -q -U peft
from google.colab import drive; drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/llm_sft_data'
ADAPTER = '/content/drive/MyDrive/llm_sft_runs/sft_lora_qwen_rag/E_alpha64/adapter'
BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
# norag: qwen_norag_questions.jsonl -> qwen_norag.jsonl ; rag: qwen_rag_prompts.jsonl -> qwen_rag.jsonl
IN_FILE, OUT_FILE, COND = 'qwen_rag_prompts.jsonl', 'qwen_rag.jsonl', 'qwen_rag'
TEMPERATURE = 0.3   # agent default 0.8; set OUT_FILE / COND to match (e.g. 'qwen_norag_t03.jsonl', 'qwen_norag_t03')


In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
tok = AutoTokenizer.from_pretrained(BASE); tok.padding_side = 'left'
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16).to('cuda')
model = PeftModel.from_pretrained(model, ADAPTER).eval()
rows = [json.loads(l) for l in open(f'{DRIVE_DIR}/{IN_FILE}')]
def msgs(r):   # rag rows carry the agent's own prompt; norag = sprout.config.system_prompt + question
    if 'messages' in r:
        return r['messages']
    return [{'role': 'system', 'content': f"You are talking to a child aged {r['age']}. Generate a friendly response with age-appropriate knowledge."},
            {'role': 'user', 'content': r['instruction']}]
GEN = dict(max_new_tokens=160, do_sample=True, temperature=TEMPERATURE, top_p=0.9, top_k=30, repetition_penalty=1.05)
torch.manual_seed(42); preds = []
with torch.no_grad():
    for i in range(0, len(rows), 16):
        texts = [tok.apply_chat_template(msgs(r), tokenize=False, add_generation_prompt=True) for r in rows[i:i+16]]
        enc = tok(texts, return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
        out = model.generate(**enc, **GEN, pad_token_id=tok.pad_token_id or tok.eos_token_id)
        preds += [p.strip() for p in tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]
with open(f'{DRIVE_DIR}/{OUT_FILE}', 'w') as f:
    for r, p in zip(rows, preds):
        f.write(json.dumps(dict({k: v for k, v in r.items() if k != 'messages'}, prediction=p, condition=COND), ensure_ascii=False) + '\n')
for r, p in list(zip(rows, preds))[:5]:
    print(r['age'], '|', r['instruction'], '\n ->', p, '\n')